In [491]:
import pydicom
import numpy as np
import radiomics
from radiomics import featureextractor
import glob
import os
import pandas as pd
from pandas import DataFrame
import csv
import SimpleITK as sitk
import matplotlib.pylab as plt

In [492]:
paramPath = 'Params_tol_0_0001.yaml'

In [ ]:
############################################### da cambiare per ogni valore di dose e macchina
data_path = '../../GE_LUNG_4_ASIR40_Shape_175/' #percorso cartella
mask_path = '../../GE_lung/GE_LUNG4_Asir40_Shape175_mask/' #percorso maschere di tutta la cartella
images_fnames = glob.glob(os.path.join(data_path,'*.dcm'))

############################################### da cambiare per ogni inserto
mask_name = 'water' #SPECIFICARE L'INSERTO: acr, air, del, LDPE, PMP, pol, tef, water
###############################################
masks_fnames = glob.glob(os.path.join(mask_path,'mask_' + mask_name + '*.dcm'))  

############################################### da cambiare per ogni dose e macchina
cwdfeatures = '../../Features/GE_LUNG4_ASIR40/Shape_175/' #cartella dove salvare le features
###############################################

#csv_file = "Features/Catphan/GE_LUNG7/'delrin.csv"
csv_file = glob.glob(os.path.join(cwdfeatures, mask_name + '*.csv')) 

In [494]:
print(images_fnames)
print(masks_fnames)
print(csv_file)

['../../Dati_mod/GE_lung/GE_LUNG_4_ASIR40_Shape_175/GE_LUNG_4_ASIR40_1.dcm']
['../../Dati_mod/GE_lung/GE_LUNG4_Asir40_Shape175_mask/mask_water.dcm']
[]


In [495]:
print(len(images_fnames))

1


In [496]:
print(masks_fnames)

['../../Dati_mod/GE_lung/GE_LUNG4_Asir40_Shape175_mask/mask_water.dcm']


In [497]:
extractor = featureextractor.RadiomicsFeatureExtractor(paramPath)#, correctMask=True)    #verificare parametri
verbose = True
if verbose:
    print('Extraction parameters:\n\t', extractor.settings)
    print('Enabled filters:\n\t', extractor.enabledImagetypes)
    print('Enabled features:\n\t', extractor.enabledFeatures)

Extraction parameters:
	 {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True, 'binWidth': 25, 'weightingNorm': None, 'geometryTolerance': 0.0001}
Enabled filters:
	 {'Original': {}}
Enabled features:
	 {'firstorder': [], 'glcm': ['Autocorrelation', 'JointAverage', 'ClusterProminence', 'ClusterShade', 'ClusterTendency', 'Contrast', 'Correlation', 'DifferenceAverage', 'DifferenceEntropy', 'DifferenceVariance', 'JointEnergy', 'JointEntropy', 'Imc1', 'Imc2', 'Idm', 'Idmn', 'Id', 'Idn', 'InverseVariance', 'MaximumProbability', 'SumEntropy', 'SumSquares'], 'glrlm': None, 'glszm': None, 'gldm': None, 'ngtdm': None}


In [498]:
#feature extraction
def_features = [{},{}]
restotal1=[0 for i in range(len(images_fnames))]
restotal=[]


for i in range(len(images_fnames)):
    
    imageName = images_fnames[i]
    maskNamepre = masks_fnames[0]
    filename = os.path.splitext(os.path.basename(imageName))[0]
    print(filename)
    
    num_acquisition = int(filename[-1])
    if num_acquisition > 3 or num_acquisition == 0:
        pass
    
    else:
   
        print("Calculating features tef")
        print(imageName)
        print(maskNamepre)
        image = pydicom.dcmread(imageName)
        seg = pydicom.dcmread(maskNamepre)
        # Convert the pydicom image to SimpleITK format
        image_sitk = sitk.GetImageFromArray(image.pixel_array)
        # Convert the pydicom segmentation mask to SimpleITK format
        seg_sitk = sitk.GetImageFromArray(seg.pixel_array)
        first_underscore_index = filename.find("_")
        second_underscore_index = filename.find("_", first_underscore_index + 1)
        third_underscore_index = filename.find("_", second_underscore_index + 1)
        fourth_underscore_index = filename.find("_", third_underscore_index + 1)
    
        result = extractor.execute(image_sitk, seg_sitk)
        result['image_ID'] = filename
        result['Manufacturer'] = image.Manufacturer
        result['Slice Thickness'] = image.SliceThickness
        result['Pixel_Spacing'] = image.PixelSpacing
        result['Kernel'] = filename[first_underscore_index +1 : second_underscore_index]
        result['Dose'] = filename[second_underscore_index + 1 : third_underscore_index]
        result['ASIR'] = filename[third_underscore_index +1 : fourth_underscore_index]
        result['Num_Acquisition'] = filename[-1]
        feat = []
        valore = []
        restotal.append(result)
        for key, value in result.items():
            feat.append(key)
            valore.append(value)
        def_features = {'features':feat, filename:valore}
        resframe=pd.DataFrame(def_features)
        restotal1[i]=pd.DataFrame(resframe)
        resframe.to_csv(os.path.join(cwdfeatures, mask_name + filename +'.csv'), index = False)  
        print('Computed ',i, '\n')
    

GE_LUNG_4_ASIR40_1
Calculating features tef
../../Dati_mod/GE_lung/GE_LUNG_4_ASIR40_Shape_175/GE_LUNG_4_ASIR40_1.dcm
../../Dati_mod/GE_lung/GE_LUNG4_Asir40_Shape175_mask/mask_water.dcm
Computed  0 



In [499]:
csv_columns =  list(restotal[0].keys())

In [500]:
# Se non ci sono file corrispondenti, creane uno
'''
if not csv_file:
    csv_file = os.path.join(cwdfeatures, mask_name + '.csv')
else:
    csv_file = csv_file[0]  # Seleziona il primo file corrispondente
'''
csv_file = os.path.join(cwdfeatures, mask_name + '.csv')

with open(csv_file, 'w') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns, extrasaction='ignore' )
    writer.writeheader()
    for data in restotal:
        writer.writerow(data)

In [501]:
pd.set_option('display.max_columns', None)
GGO = pd.read_csv(csv_file)
#print(GGO)
GGO

,diagnostics_Versions_PyRadiomics,diagnostics_Versions_Numpy,diagnostics_Versions_SimpleITK,diagnostics_Versions_PyWavelet,diagnostics_Versions_Python,diagnostics_Configuration_Settings,diagnostics_Configuration_EnabledImageTypes,diagnostics_Image-original_Hash,diagnostics_Image-original_Dimensionality,diagnostics_Image-original_Spacing,diagnostics_Image-original_Size,diagnostics_Image-original_Mean,diagnostics_Image-original_Minimum,diagnostics_Image-original_Maximum,diagnostics_Mask-original_Hash,diagnostics_Mask-original_Spacing,diagnostics_Mask-original_Size,diagnostics_Mask-original_BoundingBox,diagnostics_Mask-original_VoxelNum,diagnostics_Mask-original_VolumeNum,diagnostics_Mask-original_CenterOfMassIndex,diagnostics_Mask-original_CenterOfMass,original_firstorder_10Percentile,original_firstorder_90Percentile,original_firstorder_Energy,original_firstorder_Entropy,original_firstorder_InterquartileRange,original_firstorder_Kurtosis,original_firstorder_Maximum,original_firstorder_MeanAbsoluteDeviation,original_firstorder_Mean,original_firstorder_Median,original_firstorder_Minimum,original_firstorder_Range,original_firstorder_RobustMeanAbsoluteDeviation,original_firstorder_RootMeanSquared,original_firstorder_Skewness,original_firstorder_TotalEnergy,original_firstorder_Uniformity,original_firstorder_Variance,original_glcm_Autocorrelation,original_glcm_JointAverage,original_glcm_ClusterProminence,original_glcm_ClusterShade,original_glcm_ClusterTendency,original_glcm_Contrast,original_glcm_Correlation,original_glcm_DifferenceAverage,original_glcm_DifferenceEntropy,original_glcm_DifferenceVariance,original_glcm_JointEnergy,original_glcm_JointEntropy,original_glcm_Imc1,original_glcm_Imc2,original_glcm_Idm,original_glcm_Idmn,original_glcm_Id,original_glcm_Idn,original_glcm_InverseVariance,original_glcm_MaximumProbability,original_glcm_SumEntropy,original_glcm_SumSquares,original_glrlm_GrayLevelNonUniformity,original_glrlm_GrayLevelNonUniformityNormalized,original_glrlm_GrayLevelVariance,original_glrlm_HighGrayLevelRunEmphasis,original_glrlm_LongRunEmphasis,original_glrlm_LongRunHighGrayLevelEmphasis,original_glrlm_LongRunLowGrayLevelEmphasis,original_glrlm_LowGrayLevelRunEmphasis,original_glrlm_RunEntropy,original_glrlm_RunLengthNonUniformity,original_glrlm_RunLengthNonUniformityNormalized,original_glrlm_RunPercentage,original_glrlm_RunVariance,original_glrlm_ShortRunEmphasis,original_glrlm_ShortRunHighGrayLevelEmphasis,original_glrlm_ShortRunLowGrayLevelEmphasis,original_glszm_GrayLevelNonUniformity,original_glszm_GrayLevelNonUniformityNormalized,original_glszm_GrayLevelVariance,original_glszm_HighGrayLevelZoneEmphasis,original_glszm_LargeAreaEmphasis,original_glszm_LargeAreaHighGrayLevelEmphasis,original_glszm_LargeAreaLowGrayLevelEmphasis,original_glszm_LowGrayLevelZoneEmphasis,original_glszm_SizeZoneNonUniformity,original_glszm_SizeZoneNonUniformityNormalized,original_glszm_SmallAreaEmphasis,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_gldm_DependenceEntropy,original_gldm_DependenceNonUniformity,original_gldm_DependenceNonUniformityNormalized,original_gldm_DependenceVariance,original_gldm_GrayLevelNonUniformity,original_gldm_GrayLevelVariance,original_gldm_HighGrayLevelEmphasis,original_gldm_LargeDependenceEmphasis,original_gldm_LargeDependenceHighGrayLevelEmphasis,original_gldm_LargeDependenceLowGrayLevelEmphasis,original_gldm_LowGrayLevelEmphasis,original_gldm_SmallDependenceEmphasis,original_gldm_SmallDependenceHighGrayLevelEmphasis,original_gldm_SmallDependenceLowGrayLevelEmphasis,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength,image_ID,Manufacturer,Slice Thickness,Pixel_Spacing,Kernel,Dose,ASIR,Num_Acquisition
0,v3.0.1,1.26.2,2.3.1-g42ce2,1.4.1,3.10.13,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Orig

In [502]:
pd.set_option('display.max_columns', None)
GGO.columns

Index(['diagnostics_Versions_PyRadiomics', 'diagnostics_Versions_Numpy',
       'diagnostics_Versions_SimpleITK', 'diagnostics_Versions_PyWavelet',
       'diagnostics_Versions_Python', 'diagnostics_Configuration_Settings',
       'diagnostics_Configuration_EnabledImageTypes',
       'diagnostics_Image-original_Hash',
       'diagnostics_Image-original_Dimensionality',
       'diagnostics_Image-original_Spacing',
       ...
       'original_ngtdm_Contrast', 'original_ngtdm_Strength', 'image_ID',
       'Manufacturer', 'Slice Thickness', 'Pixel_Spacing', 'Kernel', 'Dose',
       'ASIR', 'Num_Acquisition'],
      dtype='object', length=121)